# Iris Species Classification — Model Selection & Evaluation

**Goal:** build, compare, tune, and evaluate classification models for `iris.csv`.

**Constraints respected:** no exploratory data analysis (EDA) and **no feature scaling**. The notebook does not use `StandardScaler`, `MinMaxScaler`, normalization, or any other scaling transformation.

### Workflow
1. Load the CSV and identify the target column.
2. Perform only the checks required to make the dataset usable.
3. Create a stratified train/test split.
4. Compare several classifiers using stratified cross-validation on the training set.
5. Tune a strong tree-ensemble candidate using cross-validation.
6. Evaluate the selected model once on the held-out test set.
7. Inspect errors and save the fitted model.

> Put `iris.csv` in the same directory as this notebook. If automatic target detection is wrong, set `TARGET_COLUMN` in the next cell explicitly.

In [ ]:
# Install dependencies if needed (uncomment in a fresh environment)
# %pip install -U pandas numpy scikit-learn matplotlib seaborn joblib

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, classification_report, ConfusionMatrixDisplay,
    confusion_matrix, matthews_corrcoef
)
from sklearn.inspection import permutation_importance
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

RANDOM_STATE = 42
CSV_PATH = Path("iris.csv")
TARGET_COLUMN = None  # e.g. "species"; None = infer common target names or use last column

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH.resolve()}. Place iris.csv beside this notebook "
        "or update CSV_PATH."
    )

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")

## 1. Prepare features and target

This section is intentionally limited to loading, target selection, and basic validity checks. It is not an EDA section.

In [ ]:
# Prefer common Iris target-column names; otherwise use the final column.
if TARGET_COLUMN is None:
    candidates = ["species", "variety", "class", "target", "label"]
    target_matches = [c for c in df.columns if str(c).strip().lower() in candidates]
    target_col = target_matches[0] if target_matches else df.columns[-1]
else:
    target_col = TARGET_COLUMN
    if target_col not in df.columns:
        raise KeyError(f"TARGET_COLUMN={target_col!r} not found. Available: {list(df.columns)}")

# Remove common exported index columns only (not actual measurements).
index_like = [
    c for c in df.columns
    if str(c).strip().lower() in {"id", "index", "unnamed: 0"}
]
feature_cols = [c for c in df.columns if c != target_col and c not in index_like]

if not feature_cols:
    raise ValueError("No feature columns remain. Check TARGET_COLUMN and the CSV structure.")

X = df[feature_cols].copy()
y = df[target_col].copy()

# Convert features to numeric; invalid text becomes missing and is imputed inside each CV fold.
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Remove rows without a known target; classification cannot train on these.
valid_target = y.notna()
X, y = X.loc[valid_target], y.loc[valid_target]

if y.nunique() < 2:
    raise ValueError("The target must contain at least two classes.")
if X.isna().all(axis=None):
    raise ValueError("All feature values are missing or non-numeric.")

print("Target column:", target_col)
print("Feature columns:", list(X.columns))
print("Rows available for modelling:", len(X))
print("Number of classes:", y.nunique())

## 2. Train/test split

The test set is held back until final evaluation. Stratification preserves class proportions in both partitions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")

## 3. Define candidate models

No scaling is applied. Median imputation is included inside each model pipeline so missing feature values are handled without leaking information across cross-validation folds. Models that require fully numeric inputs are given the same numeric feature matrix.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1,
        class_weight="balanced"
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1,
        class_weight="balanced"
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "SVC (RBF)": SVC(C=1.0, kernel="rbf"),
    "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=5),
    "Gaussian Naive Bayes": GaussianNB(),
    "Linear Discriminant Analysis": LinearDiscriminantAnalysis()
}

def make_pipeline(estimator):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", clone(estimator))
    ])

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro"
}

## 4. Cross-validated model comparison

All scores below are calculated using **training data only**. Macro F1 gives each class equal weight and is useful alongside accuracy.

In [ ]:
cv_rows = []
fitted_candidates = {}

for name, estimator in models.items():
    pipe = make_pipeline(estimator)
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring=scoring,
        n_jobs=-1, return_train_score=False,
        error_score="raise"
    )
    cv_rows.append({
        "Model": name,
        "CV Accuracy (mean)": scores["test_accuracy"].mean(),
        "CV Accuracy (std)": scores["test_accuracy"].std(),
        "CV Balanced Accuracy": scores["test_balanced_accuracy"].mean(),
        "CV Macro F1": scores["test_f1_macro"].mean()
    })
    fitted_candidates[name] = pipe

cv_results = pd.DataFrame(cv_rows).sort_values(
    ["CV Macro F1", "CV Accuracy (mean)"], ascending=False
).reset_index(drop=True)

display(cv_results.style.format({
    "CV Accuracy (mean)": "{:.4f}",
    "CV Accuracy (std)": "{:.4f}",
    "CV Balanced Accuracy": "{:.4f}",
    "CV Macro F1": "{:.4f}"
}))

## 5. Tune the leading tree-ensemble model

Extra Trees is a strong choice for a small tabular dataset and does not require scaling. We tune it with a compact grid using training data only. The best estimator is selected by macro F1.

In [ ]:
extra_trees_pipe = make_pipeline(
    ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
)

param_grid = {
    "model__n_estimators": [200, 400, 700],
    "model__max_features": ["sqrt", None],
    "model__min_samples_split": [2, 4, 6],
    "model__min_samples_leaf": [1, 2],
    "model__max_depth": [None, 3, 5, 8]
}

search = GridSearchCV(
    estimator=extra_trees_pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)
search.fit(X_train, y_train)

print("Best cross-validation macro F1:", round(search.best_score_, 4))
print("Best parameters:")
for key, value in search.best_params_.items():
    print(f"  {key}: {value}")

best_model = search.best_estimator_

## 6. Final held-out test evaluation

The test set is used here for the first time. Do not repeatedly tune against these results; doing so would turn the test set into training feedback.

In [ ]:
y_pred = best_model.predict(X_test)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "Matthews correlation coefficient": matthews_corrcoef(y_test, y_pred)
}

print("Selected model: Tuned Extra Trees")
for metric, value in test_metrics.items():
    print(f"{metric:35s}: {value:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 7. Confusion matrix

The confusion matrix shows which species are being confused. It is an evaluation diagnostic, not exploratory data analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    cmap="Blues",
    values_format="d",
    xticks_rotation=45,
    ax=ax
)
ax.set_title("Tuned Extra Trees — Test Confusion Matrix")
plt.tight_layout()
plt.show()

## 8. Feature importance

Permutation importance measures the drop in held-out accuracy when a feature is shuffled. It is model-agnostic and works with the complete pipeline. Interpret these values cautiously because the test set is small.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="accuracy",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance mean": perm.importances_mean,
    "Importance std": perm.importances_std
}).sort_values("Importance mean", ascending=False)

display(importance.style.format({
    "Importance mean": "{:.4f}",
    "Importance std": "{:.4f}"
}))

fig, ax = plt.subplots(figsize=(7, 4))
plot_data = importance.sort_values("Importance mean")
ax.barh(plot_data["Feature"], plot_data["Importance mean"], xerr=plot_data["Importance std"])
ax.set_xlabel("Decrease in test accuracy after permutation")
ax.set_title("Permutation Feature Importance")
plt.tight_layout()
plt.show()

## 9. Inspect misclassified test examples

In [ ]:
misclassified = X_test.loc[y_test.index[y_test.to_numpy() != y_pred]]
if len(misclassified):
    errors = misclassified.copy()
    errors["Actual"] = y_test.loc[misclassified.index]
    errors["Predicted"] = pd.Series(y_pred, index=X_test.index).loc[misclassified.index]
    display(errors)
else:
    print("No misclassified examples in the held-out test set.")

## 10. Save the trained model

The saved pipeline includes the imputer and classifier. It does **not** include any scaler because no scaling was performed.

In [ ]:
import joblib

MODEL_PATH = Path("iris_extra_trees_model.joblib")
joblib.dump(best_model, MODEL_PATH)
print(f"Saved model to: {MODEL_PATH.resolve()}")

# Example: load and predict new samples with the same feature columns.
loaded_model = joblib.load(MODEL_PATH)
sample_predictions = loaded_model.predict(X_test.iloc[:5])
print("Example predictions:", sample_predictions)

## 11. Conclusion

- The candidate models were compared using stratified 5-fold cross-validation on the training partition.
- Extra Trees hyperparameters were selected using training-only cross-validation and macro F1.
- The final model was evaluated on a separate stratified test partition.
- No scaling or normalization was performed at any stage.
- The fitted preprocessing-and-model pipeline was saved as `iris_extra_trees_model.joblib`.

**Reproducibility:** the random seed is fixed at `42`. If you change the dataset, verify the target column and feature columns before interpreting the results.